# §9A.3 — EIS Process Reward Model (PRM) Training (v2, bf16 — no bitsandbytes)

Trains a stepwise binary classifier on the §9A.2 graded stepwise labels (N=973, 95 qids).

**Why bf16 not 8-bit**: Colab 2025 (Python 3.12) removed `triton.ops`, which `bitsandbytes==0.44.1` imports eagerly from `peft.get_peft_model`. Qwen2.5-1.5B in **bf16** is only ~3 GB VRAM — fits T4/L4 with tons of headroom, no need for 8-bit.

**Target**: for a cumulative CoT prefix ending at step k, predict P(step-k is `good_strong` ∪ `good_weak`).
- positive = `good_strong` (1) + `good_weak` (105) = **106**
- negative = `neutral` (747) + `bad` (1) = **748**
- dropped  = `disagree` (119)

**Model**: `Qwen/Qwen2.5-1.5B-Instruct` + LoRA (r=16) + scalar regression head. ~10–30 min on T4/L4/A100.

**Gate** (§9A.3): held-out (by qid) AUROC ≥ 0.70 → PASS.

## 0. Runtime & GPU sanity

In [ ]:
!nvidia-smi | head -20
import torch
print('cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## 1. Install dependencies (no bitsandbytes!)

Deliberately skipping `bitsandbytes` — its 0.44 wheel on Colab triggers `ModuleNotFoundError: triton.ops` the moment `peft.get_peft_model` is called.

The `scikit-learn / umap-learn / hdbscan` dependency conflict warnings are harmless — we only use `sklearn.metrics.roc_auc_score`.

In [ ]:
# Clean up any leftover bnb from a previous run.
!pip uninstall -y bitsandbytes 2>/dev/null
%pip install -q -U transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0 scikit-learn==1.5.2
import transformers, peft, accelerate, datasets
print('transformers', transformers.__version__,
      '| peft', peft.__version__,
      '| accelerate', accelerate.__version__)
# Sanity: bnb should be gone.
try:
    import bitsandbytes  # noqa
    print('WARNING: bitsandbytes still importable — restart kernel and rerun cell 1.')
except ModuleNotFoundError:
    print('OK: bitsandbytes not installed (intended).')

## 2. Upload training data

Upload both files in one shot: select `real_N128_graded_20260421.jsonl` **and** `cot_deepseek-r1_N128.jsonl`.

In [ ]:
from google.colab import files
up = files.upload()
print('uploaded:', list(up))

## 3. Build stepwise training set (qid-level split)

In [ ]:
import json, random, re
from collections import Counter

with open('real_N128_graded_20260421.jsonl', encoding='utf-8') as f:
    graded = [json.loads(l) for l in f]

cot_rows = {}
with open('cot_deepseek-r1_N128.jsonl', encoding='utf-8') as f:
    for l in f:
        r = json.loads(l)
        cot_rows[r['qid']] = r

POS = {'good_strong', 'good_weak'}
NEG = {'neutral', 'bad'}
DROP = {'disagree'}

def step_prefix(qid, step_idx, fallback_text=None):
    r = cot_rows.get(qid)
    if r is None:
        return fallback_text
    steps = r.get('steps') or r.get('cot_steps')
    if steps is None:
        txt = r.get('answer') or r.get('reasoning') or ''
        parts = re.split(r'(?=Step\s*\d+[:：])', txt)
        steps = [p.strip() for p in parts if p.strip()]
    if not steps or step_idx >= len(steps):
        return fallback_text
    return '\n\n'.join(steps[:step_idx + 1])

rows = []
for g in graded:
    if g['consensus_label'] in DROP:
        continue
    y = 1 if g['consensus_label'] in POS else 0
    prefix = step_prefix(g['qid'], g['step_idx'], fallback_text=g.get('step_text',''))
    if not prefix:
        continue
    rows.append({'qid': g['qid'], 'step_idx': g['step_idx'],
                 'prefix': prefix, 'label': y})
print('usable rows:', len(rows), '| label dist:', Counter(r['label'] for r in rows))

qids = sorted({r['qid'] for r in rows})
random.seed(42); random.shuffle(qids)
n_eval = max(8, len(qids) // 5)
eval_qids = set(qids[:n_eval])
train = [r for r in rows if r['qid'] not in eval_qids]
evl   = [r for r in rows if r['qid'] in eval_qids]
print(f'train={len(train)} (pos={sum(r["label"] for r in train)})  '
      f'eval={len(evl)} (pos={sum(r["label"] for r in evl)})  '
      f'eval qids={len(eval_qids)}')

## 4. Tokenize

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_LEN  = 2048

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def encode(batch):
    e = tok(batch['prefix'], truncation=True, max_length=MAX_LEN,
            padding='max_length')
    e['labels'] = [float(x) for x in batch['label']]
    return e

ds_tr = Dataset.from_list(train).map(
    encode, batched=True, remove_columns=['qid','step_idx','prefix','label'])
ds_ev = Dataset.from_list(evl).map(
    encode, batched=True, remove_columns=['qid','step_idx','prefix','label'])
print(ds_tr, ds_ev)

## 5. Build model: Qwen2.5-1.5B (bf16) + LoRA + scalar head

In [ ]:
import torch, torch.nn as nn
from transformers import AutoModel
from peft import LoraConfig, get_peft_model

base = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.bfloat16, device_map='auto',
)
# Freeze base; LoRA will mark its adapters trainable.
for p in base.parameters():
    p.requires_grad = False

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    task_type='FEATURE_EXTRACTION',
)
base = get_peft_model(base, lora)
base.print_trainable_parameters()

HIDDEN = base.config.hidden_size

class PRM(nn.Module):
    def __init__(self, backbone, hidden):
        super().__init__()
        self.bb = backbone
        # Scalar head in fp32 for numerically stable BCE.
        self.head = nn.Linear(hidden, 1)
    def forward(self, input_ids, attention_mask, labels=None):
        out = self.bb(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        seq_len = attention_mask.sum(dim=1) - 1
        h_last = h[torch.arange(h.size(0)), seq_len].to(torch.float32)
        logit = self.head(h_last).squeeze(-1)
        if labels is None:
            return logit
        loss = nn.functional.binary_cross_entropy_with_logits(
            logit, labels.float(),
            pos_weight=torch.tensor(7.0, device=logit.device),
        )
        return {'loss': loss, 'logits': logit}

model = PRM(base, HIDDEN)
emb_dev = next(model.bb.parameters()).device
model.head = model.head.to(emb_dev).to(torch.float32)
print('backbone dev:', emb_dev, '| head dev:', next(model.head.parameters()).device)

## 6. Train (manual loop; LoRA adapters + scalar head only)

In [ ]:
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import roc_auc_score

def collate(batch):
    return {k: torch.tensor([b[k] for b in batch])
            for k in ['input_ids', 'attention_mask', 'labels']}

BATCH  = 2   # T4 safe; bump to 4 on L4, 8 on A100
EPOCHS = 3
LR     = 2e-4

trainable = [p for p in model.parameters() if p.requires_grad]
print(f'trainable params: {sum(p.numel() for p in trainable):,}')
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.01)
dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True,  collate_fn=collate)
dl_ev = DataLoader(ds_ev, batch_size=BATCH, shuffle=False, collate_fn=collate)

def eval_auroc():
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for b in dl_ev:
            b = {k: v.to(emb_dev) for k, v in b.items()}
            out = model(b['input_ids'], b['attention_mask'])
            scores.extend(out.float().cpu().numpy().tolist())
            labels.extend(b['labels'].cpu().numpy().tolist())
    try:
        return float(roc_auc_score(labels, scores))
    except Exception:
        return float('nan')

history = []
for ep in range(EPOCHS):
    model.train()
    losses = []
    for step, b in enumerate(dl_tr):
        b = {k: v.to(emb_dev) for k, v in b.items()}
        out = model(**b)
        out['loss'].backward()
        opt.step(); opt.zero_grad()
        losses.append(float(out['loss']))
        if step % 50 == 0:
            print(f'ep{ep} step{step}/{len(dl_tr)} loss={np.mean(losses[-50:]):.3f}')
    auroc = eval_auroc()
    history.append({'epoch': ep, 'mean_loss': float(np.mean(losses)),
                    'eval_auroc': auroc})
    print(f'>>> ep{ep} train-loss={np.mean(losses):.3f}  eval-AUROC={auroc:.3f}')
print('history:', history)

## 7. §9A.3 gate + save artifacts

In [ ]:
final_auroc = history[-1]['eval_auroc']
GATE = 0.70
print(f'§9A.3 gate: eval AUROC = {final_auroc:.3f}  '
      f'threshold ≥ {GATE}  → {"PASS" if final_auroc >= GATE else "FAIL"}')

import os, json
os.makedirs('prm_v1_lora', exist_ok=True)
model.bb.save_pretrained('prm_v1_lora')
tok.save_pretrained('prm_v1_lora')
torch.save(model.head.state_dict(), 'prm_v1_head.pt')
with open('prm_v1_metrics.json', 'w', encoding='utf-8') as f:
    json.dump({'history': history, 'final_auroc': final_auroc,
                'gate_threshold': GATE, 'gate_pass': final_auroc >= GATE,
                'n_train': len(train), 'n_eval': len(evl),
                'n_pos_train': int(sum(r['label'] for r in train))},
               f, indent=2)
!ls -lh prm_v1_lora prm_v1_head.pt prm_v1_metrics.json

## 8. Push artifacts to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pvgap_prm_v1
!cp -r prm_v1_lora prm_v1_head.pt prm_v1_metrics.json /content/drive/MyDrive/pvgap_prm_v1/
!ls /content/drive/MyDrive/pvgap_prm_v1/

## 9. Inference sanity — reload artifacts and score a sample

Mirror this pattern later in `pvgap_experiment/src/prm_scorer.py` to wire `weaver_signals.extract_w1_prm(prm_model=...)`.

In [ ]:
model.eval()
demo = ds_ev[0]
enc = {k: torch.tensor([demo[k]], device=emb_dev) for k in ['input_ids','attention_mask']}
with torch.no_grad():
    logit = model(enc['input_ids'], enc['attention_mask'])
    prob  = torch.sigmoid(logit).item()
print(f'true label={demo["labels"]:.0f}  pred prob={prob:.3f}')